NOTE : Day 56B mein dekha tha ki imbalanced data mein model majority class ko favour karta hai aur minority (churned) customers ko miss kar deta hai. Do common fixes hain:

- class_weight='balanced': data ko touch nahi karta. Model ko bolta hai ki minority class ki galti ko zyada penalty do.
- SMOTE: minority class ke naye synthetic samples banata hai (existing minority points ke beech interpolate karke), taaki training data balanced ho jaye.

Teen rules jo aaj sabse zaroori hain:

1. SMOTE sirf training data par lagta hai. Test set aur validation fold kabhi resample nahi hote.
2. CV ke andar SMOTE chalane ke liye imblearn.pipeline.Pipeline use karna padta hai (normal sklearn Pipeline samplers support nahi karti).
3. Compare hamesha Recall, F1 aur ROC-AUC se karo, accuracy se nahi.


1. Library install aur imports:

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

X_train = pd.read_csv('../data/processed/X_train.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()

num_cols = ['Recency', 'Frequency', 'Monetary_log', 'AvgOrderValue', 'Tenure', 'UniqueProducts']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

2. Pehle imbalance dobara dekho (Day 62 ke naye label par):

In [3]:
counts = y_train.value_counts()
print(counts)
print(f"Imbalance ratio: {counts.max() / counts.min():.2f} : 1")

Churned
1    2393
0    1833
Name: count, dtype: int64
Imbalance ratio: 1.31 : 1


Yeh number Day 56B se alag ho sakta hai, kyunki label ab future window par bana hai. Agar ratio 1.5:1 ke aas-paas hai (mild), to dono techniques ka fark chhota dikh sakta hai, aur yeh bhi ek valid finding hai.

3. Approach A: Baseline (koi handling nahi):

In [4]:
prep = ColumnTransformer([('scale', StandardScaler(), num_cols)], remainder='passthrough')

baseline = ImbPipeline([
    ('prep', prep),
    ('model', LogisticRegression(max_iter=5000))
])
res_base = cross_validate(baseline, X_train, y_train, cv=cv, scoring=metrics)

4. Approach B: class_weight='balanced':

In [5]:
cw = ImbPipeline([
    ('prep', prep),
    ('model', LogisticRegression(max_iter=5000, class_weight='balanced'))
])
res_cw = cross_validate(cw, X_train, y_train, cv=cv, scoring=metrics)

5. Approach C: SMOTE (pipeline ke andar, har fold ke train hisse par hi):

In [6]:
smote = ImbPipeline([
    ('prep', prep),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=5000))
])
res_smote = cross_validate(smote, X_train, y_train, cv=cv, scoring=metrics)

Order dhyan se dekho: pehle scaling, phir SMOTE, phir model. SMOTE distance par based hai, isliye scaled features par kaam karna chahiye. Aur kyunki yeh pipeline ke andar hai, validation fold par SMOTE kabhi nahi lagta.

6. Teeno ka comparison table:

In [7]:
def summarize(res):
    return {m: res[f'test_{m}'].mean() for m in metrics}

comparison = pd.DataFrame({
    'Baseline': summarize(res_base),
    'class_weight=balanced': summarize(res_cw),
    'SMOTE': summarize(res_smote)
}).T.round(3)
print(comparison)

stds = pd.DataFrame({
    'Baseline': {m: res_base[f'test_{m}'].std() for m in metrics},
    'class_weight': {m: res_cw[f'test_{m}'].std() for m in metrics},
    'SMOTE': {m: res_smote[f'test_{m}'].std() for m in metrics}
}).T.round(3)
print("\nStd across folds:\n", stds)

                       accuracy  precision  recall     f1  roc_auc
Baseline                  0.730      0.744   0.797  0.770    0.794
class_weight=balanced     0.722      0.773   0.720  0.745    0.794
SMOTE                     0.722      0.774   0.718  0.745    0.793

Std across folds:
               accuracy  precision  recall     f1  roc_auc
Baseline         0.011      0.010   0.014  0.009    0.013
class_weight     0.009      0.008   0.017  0.010    0.013
SMOTE            0.009      0.008   0.018  0.010    0.012


7. Result kaise padhna hai:

- Recall badhta hai, Precision ghat sakta hai. Yeh trade-off normal hai: model zyada customers ko “churn risk” bolega, kuch galti se.
- Accuracy girna theek hai. Day 56B mein yehi samjha tha ki accuracy yahan misleading metric hai.
- ROC-AUC aksar kaafi similar rehta hai. Yeh ranking quality naapta hai aur class weights/SMOTE se zyada nahi badalta.
- Agar teeno ka F1 ek jaisa aaye, to SMOTE ki extra complexity ka koi fayda nahi. Simple class_weight rakho.

8. Business ke hisaab se choose karo:

In [8]:
print("""
Retention campaign ki cost (discount/email) kam hai, churn se loss zyada hai
-> Recall ko priority do (zyada at-risk customers pakdo, kuch false alarms chalenge).

Campaign mehengi hai (bada discount)
-> Precision ko priority do.

Dono ka balance chahiye -> F1.
""")


Retention campaign ki cost (discount/email) kam hai, churn se loss zyada hai
-> Recall ko priority do (zyada at-risk customers pakdo, kuch false alarms chalenge).

Campaign mehengi hai (bada discount)
-> Precision ko priority do.

Dono ka balance chahiye -> F1.



9. SMOTE ki limitation (interview mein poochi jaati hai): Region_* dummy columns 0/1 hain, lekin SMOTE interpolation se 0.37 jaisi fractional values bana sakta hai, jo categorical ke liye meaningless hain. Proper fix SMOTENC hai:

In [9]:
from imblearn.over_sampling import SMOTENC

feature_names = num_cols + [c for c in X_train.columns if c not in num_cols]
cat_idx = [i for i, c in enumerate(feature_names) if c.startswith('Region_')]

smotenc = ImbPipeline([
    ('prep', prep),                                  # output order: scaled num cols, phir baaki (Region_*)
    ('smote', SMOTENC(categorical_features=cat_idx, random_state=42)),
    ('model', LogisticRegression(max_iter=5000))
])
res_smotenc = cross_validate(smotenc, X_train, y_train, cv=cv, scoring=metrics)
print({m: round(res_smotenc[f'test_{m}'].mean(), 3) for m in metrics})

{'accuracy': np.float64(0.722), 'precision': np.float64(0.774), 'recall': np.float64(0.719), 'f1': np.float64(0.745), 'roc_auc': np.float64(0.793)}


Yeh optional hai. Is project mein 2 dummy columns se zyada fark nahi padega, lekin concept note kar lo.

10. Final decision README ke liye note karo:

In [10]:
best = comparison['f1'].idxmax()
print(f"Best F1 approach on CV: {best}")

Best F1 approach on CV: Baseline


README mein likho: “Imbalance ratio X:1 tha. Baseline, class_weight aur SMOTE ko 5-fold CV par compare kiya. [best approach] chuna kyunki [apna reason: recall/F1 gain ya simplicity].”

practice questions:

1. SMOTE(sampling_strategy=0.8) try karo (minority ko majority ka 80% tak laata hai, poora 1:1 nahi) aur dekho recall/precision kaise badalte hain.

Jab hum sampling_strategy=1.0 rakhte hain, toh SMOTE minority class ke samples ko badhakar majority class ke barabar (100%) kar deta hai.
Jab tum ise ghatakar 0.8 (80%) karte ho, toh minority class synthetic points ke zariye majority ka 80% banti hai.

- Recall par asar: Thoda sa kam ho sakta hai, kyunki ab model ko minority class ke data points thode kam milte hain sikhne ke liye, jisse kuch at-risk customers miss ho sakte hain.

- Precision par asar: Yeh improve/badh jaata hai.

- Kyun? Jab tum 1:1 balance karte ho, toh model bohot saare false alarms (False Positives) dene lagta hai kyunki wo har jagah churner dhoondhne ki koshish karta hai (jis se Precision girta hai). sampling_strategy=0.8 ek conservative balance deta hai jahan model aggressive hokar har customer ko churner nahi bolta, isliye precision bach jaati hai jabki recall thoda controlled rehta hai.

2. Ek galat version banao: poore X_train par SMOTE pehle lagao, phir normal cross_val_score chalao. Score jhootha achha kyun dikhta hai? (Hint: synthetic copies validation folds mein bhi pahunch jaate hain.)

Agar tumne CV pipeline ke andar SMOTE lagane ke bajaye pehle poore X_train ko SMOTE se balance kar diya aur phir cross_val_score chalaya, toh ek bhayanak Data Leakage ho jaati hai:

1. Kaise hota hai leakage: SMOTE real existing data points ke beech nearest neighbors dhoondh kar unke coordinates ke beech interpolation (synthetic copying) karta hai.

2. Jhootha score kyun milta hai: Jab tumne poore X_train par pehle SMOTE laga diya, toh jo synthetic data points bane, unka kuch hissa cross-validation ke validation fold mein bhi chala gaya. Validation fold ke paas wahi synthetic points aa gaye jo training fold ke points se milte-julte the.

3. Model jab validation fold par "exam" deta hai, toh wo actually apne hi banaye hue duplicated/synthetic clones ko pehchan raha hota hai—bilkul waise hi jaise agar teacher test mein wahi sawaal de de jo kal class mein ratwaye the! Isse accuracy, F1, aur ROC-AUC artificially 99% tak badhe hue dikhte hain, lekin jab model ko bilkul naye, real-world test data (X_test) par laoge, toh wo buri tarah fail ho jayega.


Isi liye Har transformation (Scaling, SMOTE, Imputation) hamesha Cross-Validation pipeline ke andar hona chahiye, taaki validation fold hamesha 100% unseen aur honest rahe!